# SEMTM0009 Week 2 Worksheet — computational template

**Estimating $R_0$ from data.**

The plumbing is done: data loading, rolling windows, fitting, plotting. What is
left to you is marked `# TODO`, and the questions to answer *in words* are in
the markdown cells.

A `# TODO` function returns `np.nan` until you fill it in, so you can run the
whole notebook at any time and see exactly what is still missing.

**The true $R_0$ is 5 in every scenario.** You are working out which *methods*
recover it.

Run a cell with **Shift+Enter**; run everything with *Run ▸ Run All Cells*.

---
## Setup

Read the units cell carefully. It is where most marks are lost.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit, minimize_scalar

plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "lines.linewidth": 1.8,
                     "axes.spines.top": False, "axes.spines.right": False})

# ---------------------------------------------------------------- UNITS ------
# Every rate in one formula must share one time unit.  Two sets, used in
# different questions.  Never mix them.
LIFE_EXPECTANCY_YEARS = 70
INFECTIOUS_PERIOD_DAYS = 14

MU_PER_DAY  = 1 / (LIFE_EXPECTANCY_YEARS * 365)   # Q1A, Q1B  -> work in DAYS
MU_PER_YEAR = 1 / LIFE_EXPECTANCY_YEARS           # Q1C, Q2   -> work in YEARS
GAMMA_PER_DAY = 1 / INFECTIOUS_PERIOD_DAYS

R0_TRUE = 5.0

print(f"mu  = {MU_PER_DAY:.6e} / day   = {MU_PER_YEAR:.6f} / year")
print(f"gam = {GAMMA_PER_DAY:.6f} / day")
print(f"gam + mu = {GAMMA_PER_DAY + MU_PER_DAY:.6f} / day")

mu  = 3.913894e-05 / day   = 0.014286 / year
gam = 0.071429 / day
gam + mu = 0.071468 / day


In [2]:
# ------------------------------------------------------------------ DATA -----
# Reads data/ next to this notebook if it is there (Jupyter, VS Code);
# otherwise, e.g. in Google Colab, fetches the same files from the course repo.
import os
DATA = ("data" if os.path.isdir("data") else
        "https://raw.githubusercontent.com/SEMTM0009-University-of-Bristol/notebooks/main/Week2/data")

incidence = pd.read_csv(f"{DATA}/incidence.csv")   # columns: day, incidence
ages      = pd.read_csv(f"{DATA}/ages.csv")        # column:  age_years
serology  = pd.read_csv(f"{DATA}/serology.csv")    # columns: age_years, seropositive

print("incidence:", incidence.shape, "| total cases:", int(incidence.incidence.sum()))
print("ages:      ", ages.shape, "| mean age:", round(ages.age_years.mean(), 4), "years")
print("serology:  ", serology.shape, "| seropositive:", int(serology.seropositive.sum()))
display(incidence.head(3))

incidence: (31, 2) | total cases: 2389
ages:       (44, 1) | mean age: 18.75 years
serology:   (100, 2) | seropositive: 64


,day,incidence
0,0,1
1,1,2
2,2,0


---
# Question 1A — an endemic disease

44 positive out of a community of 100,000.

**Careful:** the formulas want a *fraction*, so $I^* = 44/100000$, not 44.

In [5]:
N_COMMUNITY = 100_000
N_POSITIVE  = 44
I_STAR = N_POSITIVE / N_COMMUNITY

def R0_from_prevalence(Istar, gamma, mu=MU_PER_DAY):
    '''R0 from endemic prevalence.

    Eliminate beta between  Istar = (mu/beta)(R0 - 1)  and  R0 = beta/(gamma+mu),
    then solve for R0.  Keep EVERY term when you expand Istar*(gamma+mu).

    Sanity check: the answer must be positive and bigger than 1.  A negative
    number means the denominator is the wrong way round; 5.08 instead of
    5.09 means the I*mu term was dropped. Neither is bad data.
    '''
    # TODO: replace np.nan with your formula.
    return mu/(mu - Istar*(gamma + mu))

print("A(i)   R0 =", R0_from_prevalence(I_STAR, GAMMA_PER_DAY))

A(i)   R0 = 5.087505087505087


### A(ii) — under-reporting, both readings

There are two readings of "10% of infections are asymptomatic", and they give
very different answers. Do both.

In [6]:
ASYMPTOMATIC_RATE = 0.10
rho = 1 - ASYMPTOMATIC_RATE          # fraction who come forward

# (a) truth is 44 per 100k; you observe only rho of them
I_observed = rho * I_STAR
# (b) you observed 44 per 100k, and that IS the symptomatic rho
I_true_corrected = I_STAR / rho

print(f"(a) you would report   I* = {I_observed*1e5:7.3f} /100k  ->  R0 = "
      f"{R0_from_prevalence(I_observed, GAMMA_PER_DAY)}")
print(f"(b) corrected truth    I* = {I_true_corrected*1e5:7.3f} /100k  ->  R0 = "
      f"{R0_from_prevalence(I_true_corrected, GAMMA_PER_DAY)}")
print(f"    for comparison, naive  I* = {I_STAR*1e5:7.3f} /100k  ->  R0 = "
      f"{R0_from_prevalence(I_STAR, GAMMA_PER_DAY)}")

(a) you would report   I* =  39.600 /100k  ->  R0 = 3.6113598936815645
(b) corrected truth    I* =  48.889 /100k  ->  R0 = 9.320629660314818
    for comparison, naive  I* =  44.000 /100k  ->  R0 = 5.087505087505087


**Question.** Which reading is the situation a real epidemiologist is in? In
which direction does systematic under-reporting bias this estimator, and why is
the correction so large?

*Your answer:*

### A(iii) — an uncertain infectious period

In [7]:
for ip in [12, 14, 16]:
    print(f"1/gamma = {ip:2d} d  ->  R0 = {R0_from_prevalence(I_STAR, 1/ip)}")

1/gamma = 12 d  ->  R0 = 15.942183016261
1/gamma = 14 d  ->  R0 = 5.087505087505087
1/gamma = 16 d  ->  R0 = 3.3677404145688445


#### A(iii)(c) — the ceiling

Rewrite the estimator as
$$R_0=\Bigl(1-\frac{I^*}{\mu G}\Bigr)^{-1},\qquad G=\frac{1}{\gamma+\mu}.$$

Since $1-1/R_0<1$ always, $I^* < \mu G$ **for any $R_0$ whatsoever**. Compute
that ceiling and compare it with the observed 44 per 100,000.

In [8]:
rows = []
for ip in [16, 14, 12, 11.25, 11]:
    gamma = 1 / ip
    G = 1 / (gamma + MU_PER_DAY)
    ceiling = MU_PER_DAY * G                       # max possible I*, any R0
    rows.append({"1/gamma (d)": ip,
                 "ceiling per 100k": round(ceiling * 1e5, 3),
                 "I*/ceiling": round(I_STAR / ceiling, 4),
                 "R0": R0_from_prevalence(I_STAR, gamma)})
display(pd.DataFrame(rows))

# Where does the estimator blow up?  Solve  ceiling == I*  for the period.
pole = I_STAR / (MU_PER_DAY * (1 - I_STAR))
print(f"\nthe estimator diverges at 1/gamma = {pole:.3f} days")

,1/gamma (d),ceiling per 100k,I*/ceiling,R0
0,16.00,62.583,0.7031,3.367740
1,14.00,54.765,0.8034,5.087505
2,12.00,46.945,0.9373,15.942183
3,11.25,44.012,0.9997,3688.524590
4,11.00,43.034,1.0224,-44.563280



the estimator diverges at 1/gamma = 11.247 days


**Question.** Explain the numbers in the table. What is happening physically at
$1/\gamma = 11.25$ days, and what does the negative $R_0$ at 11 days mean?

*Your answer:*

#### A(iii)(b) — the reverse question

Which matters more to the predicted prevalence: a one-day change in the
infectious period, or a one-unit change in $R_0$?

In [9]:
def prevalence_predicted(R0, period_days, mu=MU_PER_DAY):
    '''The model's endemic prevalence, as a fraction, given R0 and 1/gamma.'''
    gamma = 1 / period_days
    return mu * (1 - 1/R0) / (gamma + mu)

R0_base, T_base = 5.0875, 14.0
dI_from_one_day = (prevalence_predicted(R0_base, T_base + 1)
                   - prevalence_predicted(R0_base, T_base))
print(f"one extra day of infectious period changes I* by "
      f"{dI_from_one_day*1e5:+.4f} per 100k")

def dR0_equivalent(R0, G):
    '''Local (derivative) change in R0 equivalent to one extra day of 1/gamma.

    Differentiate prevalence_predicted w.r.t. T and w.r.t. R0 (using mu*T << 1),
    set the two changes equal, and solve for dR0. Or derive it by hand from the
    worksheet's closed form.
    '''
    # TODO: replace np.nan with your derived formula.
    return mu*G*(1-1/R0)

G = 1 / (1/T_base + MU_PER_DAY)
dR0_local = dR0_equivalent(R0_base, G)
print(f"local (derivative) equivalent : dR0 = {dR0_local:.4f}")

r_lo, r_hi = R0_base, R0_base + 10
for _ in range(200):                          # bisection for the exact answer
    m = (r_lo + r_hi) / 2
    if prevalence_predicted(m, T_base) - prevalence_predicted(R0_base, T_base) < dI_from_one_day:
        r_lo = m
    else:
        r_hi = m
dR0_exact = r_lo - R0_base
print(f"exact equivalent              : dR0 = {dR0_exact:.4f}")

print("\nhow the local answer scales with R0:")
for r in [2, 5, 10, 20]:
    print(f"  R0 = {r:2d}: one day of 1/gamma == {dR0_equivalent(r, G):7.3f} units of R0")

one extra day of infectious period changes I* by +3.1410 per 100k


NameError: name 'mu' is not defined

**Question.** Which has the greater impact? What happens as $R_0$ grows? And
why is the *exact* equivalent larger than the derivative-based one? (Hint: look
again at the ceiling table.)

*Your answer:*

---
# Question 1B — a novel invasive disease

Daily incidence from the start of an outbreak. **This is incidence $i(t)$, not
prevalence $I(t)$.**

In [ ]:
def incidence_to_prevalence(inc, period_days):
    '''Prevalence = rolling sum of incidence over one infectious period.

    Everyone infected within the last 1/gamma days is still infectious, so the
    window's FORWARD edge is the current day (a trailing window).
    min_periods=1 lets the first few days through with a partial window --
    which is exactly why early prevalence is underestimated.
    '''
    w = int(round(period_days))
    return pd.Series(inc).rolling(w, min_periods=1).sum().to_numpy()


def fit_growth_rate(prevalence, p0=(1.0, 0.3)):
    '''Least-squares fit of I(t) = I0 exp(r t).  Returns (I0, r).'''
    t = np.arange(len(prevalence), dtype=float)
    (I0, r), _ = curve_fit(lambda t, I0, r: I0 * np.exp(r * t),
                           t, prevalence, p0=p0, maxfev=40000)
    return I0, r


def R0_from_growth_rate(r, gamma, mu=MU_PER_DAY):
    '''R0 from the early exponential growth rate.

    From the disease-free equilibrium,  r = (gamma + mu)(R0 - 1).
    '''
    # TODO: replace np.nan with your formula.
    return np.nan


inc = incidence.incidence.to_numpy(dtype=float)
prev = incidence_to_prevalence(inc, INFECTIOUS_PERIOD_DAYS)
I0, r = fit_growth_rate(prev)
print(f"fitted r = {r:.4f} /day   ->  R0 = {R0_from_growth_rate(r, GAMMA_PER_DAY)}")

In [ ]:
# Look at the conversion before trusting it.
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.bar(incidence.day, inc, color="grey", alpha=0.45, label="incidence $i(t)$ (left)")
ax.set(xlabel="$t$ (days)", ylabel="incidence (new cases/day)")
ax2 = ax.twinx(); ax2.spines["top"].set_visible(False)
ax2.plot(incidence.day, prev, "o-", ms=4, color="tab:blue")
ax2.set_ylabel("prevalence (people infected)", color="tab:blue")
ax2.tick_params(axis="y", labelcolor="tab:blue")
ax.set_title("a flow and a stock")
plt.show()

### B(i)–(iv) — the whole table at once

`prop=0.5` uses only the first half of the data (part iv): the first
16 of the 31 rows, days 0–15. `report=0.9`
multiplies every day's incidence by 0.9 (part ii).

In [ ]:
def estimate_R0_from_incidence(inc, period_days, prop=1.0, report=1.0, plot=False):
    '''Full pipeline: subset -> thin -> rolling window -> fit -> convert.'''
    n = int(round(len(inc) * prop))
    x = inc[:n] * report
    prev = incidence_to_prevalence(x, period_days)
    I0, r = fit_growth_rate(prev)
    R0 = R0_from_growth_rate(r, 1 / period_days)
    if plot:
        t = np.arange(n, dtype=float)
        plt.figure(figsize=(5.2, 3.2))
        plt.plot(t, prev, "o", ms=4, label="prevalence")
        plt.plot(t, I0 * np.exp(r * t), "-", color="tab:red", label=f"fit, $r$={r:.3f}")
        plt.yscale("log"); plt.xlabel("$t$ (days)"); plt.legend(frameon=False)
        plt.title(f"prop={prop}, $1/\\gamma$={period_days} d, $R_0$={R0}")
        plt.show()
    return r, R0

rows = []
for prop, label in [(1.0, "all 31 days"), (0.5, "first half")]:
    for part, kw in [("(i)   1/g=14",       dict(period_days=14)),
                     ("(ii)  x0.9",         dict(period_days=14, report=0.9)),
                     ("(iii) 1/g=12",       dict(period_days=12)),
                     ("(iii) 1/g=16",       dict(period_days=16))]:
        r, R0 = estimate_R0_from_incidence(inc, prop=prop, **kw)
        rows.append({"data": label, "part": part, "r": round(r, 4), "R0": R0})
display(pd.DataFrame(rows))

In [ ]:
# Part (iv): the same fit on a log axis, so you can SEE where growth stops
# being exponential.
for prop in (1.0, 0.5):
    estimate_R0_from_incidence(inc, 14, prop=prop, plot=True)

**Question (ii).** Explain, *before* looking at the numbers, why multiplying
every day's incidence by 0.9 cannot change $r$. Then: a reporting *probability*
is not a deterministic multiplier — each case is reported or not at random.
Where in the time series would that randomness matter most, and what would it
do to your estimate?

*Your answer:*

**Question (iii).** $\gamma$ enters twice — as the window length and in the
final conversion. Which matters more? (Compare how much $r$ changes across the
rows with how much $R_0$ changes.)

*Your answer:*

**Question (iv).** Using *less* data made the estimate *better*. Why? And what
does that imply for a real outbreak, where you cannot see the true $R_0$?

*Your answer:*

---
# Question 1C — state of the epidemic unknown

Ages of the people coming forward. **Work in years from here on.**

In [ ]:
a = ages.age_years.to_numpy(dtype=float)
A_mean = a.mean()

def R0_from_mean_age(A, mu=MU_PER_YEAR):
    '''R0 from the mean age at infection, A = 1/(mu (R0 - 1)).

    Mind the bracket.  If you get R0 < 1 for an endemic disease, the bracket is
    where to look first.
    '''
    # TODO: replace np.nan with your formula.
    return np.nan

print(f"n = {len(a)},  A = {A_mean:.4f} years  ->  R0 = {R0_from_mean_age(A_mean)}")
print(f"note gamma does not appear anywhere above -- so C(iii) needs no work.")

### C(ii) — age-dependent ascertainment

20% of under-18s are never observed; all adults are.

In [ ]:
CHILD_AGE_LIMIT = 18
P_CHILD_MISSED  = 0.20

child, adult = a[a < CHILD_AGE_LIMIT], a[a >= CHILD_AGE_LIMIT]
n_c, n_a = len(child), len(adult)
print(f"under 18: n = {n_c}, mean = {child.mean():.4f} y")
print(f"18+     : n = {n_a}, mean = {adult.mean():.4f} y")

# TODO: the observed proportion of children once a fraction p are missed.
w = np.nan          # <-- your expression here
A_hat = w * child.mean() + (1 - w) * adult.mean()
print(f"\nreweighted: w = {w},  A_hat = {A_hat} y  ->  R0 = {R0_from_mean_age(A_hat)}")

In [ ]:
# The stochastic version: delete each child with probability 0.2 and repeat.
# The deterministic answer above should sit at the centre of this distribution.
rng = np.random.default_rng(0)
sims = np.array([R0_from_mean_age(
                    np.concatenate([child[rng.random(n_c) > P_CHILD_MISSED],
                                    adult]).mean())
                 for _ in range(5000)])
if np.isfinite(sims).all():
    plt.figure(figsize=(6, 3.2))
    plt.hist(sims, bins=45, color="tab:green", alpha=0.7)
    plt.axvline(R0_from_mean_age(A_mean), color="k", ls="--", label="no bias")
    plt.axvline(sims.mean(), color="tab:red", label=f"mean {sims.mean():.3f}")
    plt.xlabel("$\\hat R_0$"); plt.ylabel("count"); plt.legend(frameon=False)
    plt.title("random deletion of 20% of children, 5000 replicates")
    plt.show()
    print(f"mean {sims.mean():.4f},  5-95%: "
          f"[{np.percentile(sims,5):.3f}, {np.percentile(sims,95):.3f}]")
else:
    print("fill in R0_from_mean_age and w first")

**Question (a).** Argue, without computing anything, why an
age-*independent* asymptomatic rate would leave this estimate completely
unchanged.

*Your answer:*

**Question (b).** Which direction did the estimate move, and why? Compare the
size of the bias with the width of the histogram above — which is bigger, and
what does that mean for how you should report this number?

*Your answer:*

---
# Question 2 — seroprevalence

In [ ]:
age = serology.age_years.to_numpy(dtype=float)
pos = serology.seropositive.to_numpy(dtype=int)
a_neg = age[pos == 0]      # never infected -- still susceptible
b_pos = age[pos == 1]      # infected at some point

S_star = len(a_neg) / len(age)
print(f"seronegative: {len(a_neg)} of {len(age)}  ->  S* = {S_star:.4f}")

# 2A: at endemic equilibrium S* = 1/R0.
# TODO: replace np.nan.
R0_aggregate = np.nan
print(f"2A  R0 = {R0_aggregate}")

### 2B(i)–(ii) — the likelihood

$P(a)$ is the probability of *still being susceptible* at age $a$: the survival
function of the exponential distribution with rate $\lambda^*=\mu(R_0-1)$.

With $n$ seronegatives of ages $\bm a$ and $m$ seropositives of ages $\bm b$,
$$L(R_0)=\prod_i P(a_i)\prod_j\bigl(1-P(b_j)\bigr).$$
Take logs before coding it.

In [ ]:
def P_susceptible(a, R0, mu=MU_PER_YEAR):
    '''Probability of NOT having been infected by age a.'''
    # TODO: replace np.nan.  This is the survival function, 1 - CDF.
    return np.nan * np.asarray(a, dtype=float)


def neg_log_likelihood(R0, a_neg=a_neg, b_pos=b_pos, mu=MU_PER_YEAR):
    '''-log L(R0).  Watch the minus sign inside the exponential.'''
    # TODO: replace np.nan with -1 * (the log-likelihood).
    #   term 1 comes from the seronegatives, term 2 from the seropositives.
    return np.nan


res = minimize_scalar(neg_log_likelihood, bounds=(1.01, 50), method="bounded")
print(f"2B(ii)  MLE R0 = {res.x}")
print(f"        sum of seronegative ages = {a_neg.sum():.0f} years")

In [ ]:
# ALWAYS look at the likelihood before trusting an optimiser.
grid = np.linspace(1.5, 15, 400)
ll = np.array([-neg_log_likelihood(R) for R in grid])
if np.isfinite(ll).any():
    plt.figure(figsize=(6, 3.4))
    plt.plot(grid, ll)
    plt.axvline(res.x, color="tab:red", ls="--", label=f"MLE {res.x:.4f}")
    plt.axvline(R0_TRUE, color="grey", ls=":", label="truth")
    plt.xlabel("$R_0$"); plt.ylabel("log-likelihood"); plt.legend(frameon=False)
    plt.show()
else:
    print("fill in neg_log_likelihood first")

### 2B(iii) — why the two estimates differ

Plot your fitted $P(a)$ against the observed seronegative fraction in age
bands, together with the curve implied by the part-A estimate.

In [ ]:
edges = np.array([0, 5, 10, 15, 20, 30, 45, 60, 120])
ctr, frac = [], []
for age_lo, age_hi in zip(edges[:-1], edges[1:]):
    m = (age >= age_lo) & (age < age_hi)
    if m.sum():
        ctr.append(age[m].mean()); frac.append((pos[m] == 0).mean())

grid_a = np.linspace(0, 100, 300)
plt.figure(figsize=(6.6, 3.8))
plt.plot(ctr, frac, "o", ms=6, label="data (banded)")
plt.plot(grid_a, P_susceptible(grid_a, res.x), color="tab:red",
         label=f"MLE, $R_0$={res.x:.3f}")
plt.plot(grid_a, P_susceptible(grid_a, R0_aggregate), "--", color="tab:orange",
         label=f"$1/S^*$, $R_0$={R0_aggregate}")
plt.xlabel("age $a$ (years)"); plt.ylabel("$P(a)$, fraction seronegative")
plt.legend(frameon=False); plt.show()

**Question (iii).** Both estimates use the same 100 people. Explain the
difference, using the picture. What did the aggregate estimator throw away?

*Your answer:*

### 2B(iv) — the ages over 100

In [ ]:
print("ages above 100 in the file:", sorted(age[age > 100].astype(int)))

keep = age <= 100
nll_trunc = lambda R0: neg_log_likelihood(R0, a_neg=age[keep & (pos == 0)],
                                                b_pos=age[keep & (pos == 1)])
res_t = minimize_scalar(nll_trunc, bounds=(1.01, 50), method="bounded")
print(f"MLE on all data          : {res.x}")
print(f"MLE truncated at 100 y   : {res_t.x}   (drops {(~keep).sum()} people)")

**Question (iv).** What do those ages tell you about how the dataset was
generated? Does removing them change your answer, and *why not much*? Would
truncating be legitimate — and what does the same objection say about the
*model*?

*Your answer:*

---
# Across the three scenarios

**Question.** Fill in this table from your results, then answer: one method is
robust to *both* an uncertain infectious period and constant under-reporting.
Which, and why?

| | A: prevalence | B: growth rate | C: mean age |
|---|---|---|---|
| estimate at $1/\gamma=14$ d | | | |
| range over $1/\gamma\in[12,16]$ | | | |
| effect of constant under-reporting | | | |
| effect of age-dependent reporting | | | |
| needs endemic equilibrium | | | |
| needs exponential phase | | | |

**Question.** If you had to report a single $R_0$ to a public health agency,
what would you report, and what would you say about its uncertainty?

*Your answer:*